# Preprocess the INA list of files to extract the Show names and list of issues

In [10]:
import json
import os
import pandas as pd
import numpy as np
from random import randint
from datetime import datetime
from impresso_essentials.utils import ALL_MEDIA, PARTNER_TO_MEDIA
from collections import Counter

## 1. Read the two files listing each file and each notice

In [2]:
notice_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/INA/ListeDesNotices.txt"
files_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/INA/ListeDesFichiers.txt"

Note - The encodings of these files was not utf-8, we will need to write the new files in utf-8.

In [3]:
notice_df = pd.read_csv(notice_filepath, sep='\t', header=0, encoding = 'latin-1')
notice_df

,Identifiant de la notice,Thèque,Titre collection,Titre propre,Date de diffusion,Date d'enregistrement,Heure de diffusion,Titre phonogramme,Genre,Thématique,Durée
0,PHD85000118,PH (Phono),Le journal sonore de la semaine,Joseph GOEBBELS lit un texte d'Adolf HITLER su...,15/03/1939,15/03/1939,NaN,NaN,Déclaration ;,Politique ;,00:03:46
1,PHD85000121,PH (Phono),Radio journal de France,Léon Blum fait appel aux détenteurs de capitaux,17/07/1936,17/07/1936,NaN,NaN,Journal parlé ;,NaN,00:13:21
2,PHD85000672,PH (Phono),La voix de Paris,Gaston HENRY-HAYE : les projets de la nouvelle...,NaN,20/10/1935,NaN,NaN,Interview entretien ; Journal parlé ;,Tourisme ;,00:02:27
3,PHD85000963,PH (Phono),Radio journal de France,Maurice BOURDET : lecture des informations du ...,07/01/1936,07/01/1936,NaN,NaN,Journal parlé ; Papier ;,Information ;,00:05:57
4,PHD85001640,PH (Phono),Radio journal de France,Appel à la nation de Jules ROMAINS,30/10/1938,30/10/1938,NaN,NaN,Déclaration ; Journal parlé ;,Littérature ; Politique ;,00:25:00
...,...,...,...,...,...,...,...,...,...,...,...
39602,00419281,PH (Phono),Inter actualités de 19H00,Inter soir 19h00 du 29 décembre 1989,29/12/1989,29/12/1989,19:00:00,NaN,Journal parlé ;,NaN,01:01:00
39603,00419356,PH (Phono),Inter actualités de 19H00,Inter soir 19h00 du 30 décembre 1989,30/12/1989,30/12/1989,19:00:00,NaN,Journal parlé ;,NaN,00:55:00
39604,00419416,PH (Phono),Inter actualités de 19H00,Inter soir 19h00 du 31 décembre 1989,31/12/1989,31/12/1989,19:00:00,NaN,Journal parlé ;,NaN,00:15:00
39605,PHD86069295,PH (Phono),NaN,Voyage à Nancy du Maréchal PETAIN,27/05/1944,27/05/1944,NaN,NaN,Journal parlé ; Reportage ;,Information ;,00:05:52


In [4]:
files_df = pd.read_csv(files_filepath, sep='\t', header=0, encoding = 'latin-1')
files_df

,Type de notice,Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Format,Répertoire,Chemin
0,EMISSION RAD.,Titre: L'Automobile en Europe - Titre collecti...,172961,96F04705SA0005_01.MP3,00172961_96F04705SA0005_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
1,EMISSION RAD.,Titre: Le dimanche des Europeens - Titre colle...,165368,96F04705SA0001_01.MP3,00165368_96F04705SA0001_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
2,EMISSION RAD.,Titre: Enseignement education chomage et emp...,167288,96F04705SA0002_01.MP3,00167288_96F04705SA0002_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
3,EMISSION RAD.,Titre: Le service militaire - Titre collection...,169364,96F04705SA0003_01.MP3,00169364_96F04705SA0003_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
4,EMISSION RAD.,Titre: L'amour en Europe - Titre collection:C'...,171017,96F04705SA0004_01.MP3,00171017_96F04705SA0004_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
...,...,...,...,...,...,...,...,...,...,...
47725,EMISSION RAD.,Titre: Uri cite musicale barcelone n ?25 - Tit...,PHZ16062539,LD95074_01.MP3,PHZ16062539_LD95074_01___EXPORT.MP3,:::,:::,FVISIO,Université radiophonique internationale,iMPRESSO\Magazines_d'information\Universite_ra...
47726,EMISSION RAD.,Titre: Uri cite musicale barcelone n )26 - Tit...,PHZ16062540,LD95075_01.MP3,PHZ16062540_LD95075_01___EXPORT.MP3,:::,:::,FVISIO,Université radiophonique internationale,iMPRESSO\Magazines_d'information\Universite_ra...
47727,EMISSION RAD.,Titre: L'avenir du protectorat au Maroc - Titr...,PHD86044647,98INA08505PA0144_02.MP3,PHD86044647_98INA08505PA0144_02_530113_1124500...,00:53:01:13,01:12:45:00,FVISIO,08-Tribune_de_Paris_1946-1963,iMPRESSO\Journal_parle\08-Tribune_de_Paris_194...
47728,EMISSION RAD.,Titre: Inter soir 19h00 19H00 du 10 janvier 19...,580978,94F05001SA0010_01.MP3,00580978_94F05001SA0010_01_3090021_3305211_EXP...,03:09:00:21,03:30:52:11,FVISIO,15-_Inter_soir_ou_Inter_actualites_1990-_fin_j...,iMPRESSO\Journal_parle\15-_Inter_soir_ou_Inter...


### Process the files df to show correctly all columns and values

The second column: "Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion" stores the information relative to various fields. 

The goal would be to separate this into various columns. For which we need to start by identifying the format of values

In [62]:
example_val = files_df["Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion"][0]
example_val_2 = files_df["Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion"][randint(0, len(files_df))]
print(example_val)
print(example_val_2)

Titre: L'Automobile en Europe - Titre collection:C'est en France  c'est en Europe - Date:06/10/1996 - Chaine:Radio France - Heure:12:05:00 comment= notice 00172961 06/10/1996
Titre: Inter soir 19h00 du 24 novembre 1988 - Titre collection:Inter actualites de 19H00 - Date:24/11/1988 - Chaine:Radio France - Heure:19:00:00 comment= notice 00328809 24/11/1988


In [63]:
merged_cols_name = "Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion"
separated_subcols = merged_cols_name.split('-')
separated_subcols

['Titre propre',
 'Titre collection',
 'date de diffusion',
 'Chaine de diffusion',
 'Heure de diffusion',
 'Notice',
 'Date de diffusion']

In [64]:
# we have some key words which we want to use as separators and extract from the text:

sep_marker = {
    'Titre propre': "Titre: ",
    'Titre collection': "Titre collection:",
    'date de diffusion': "Date:",
    'Chaine de diffusion': "Chaine:",
    'Heure de diffusion': "Heure:",
    'Notice': " comment= notice ",
    'Date de diffusion': ""
}

In [91]:
for idx, (sep, marker) in enumerate(sep_marker.items()):
    print(idx, sep, '-->', marker)

0 Titre propre --> Titre: 
1 Titre collection --> Titre collection:
2 date de diffusion --> Date:
3 Chaine de diffusion --> Chaine:
4 Heure de diffusion --> Heure:
5 Notice -->  comment= notice 
6 Date de diffusion --> 


In [66]:
#def split_values(files_row, sep_maker=sep_marker, merged_cols=merged_cols):
def split_values_old(vals, sep_maker=sep_marker, separated_subcols=separated_subcols, merged_cols_name=merged_cols_name):
    new_vals = {}
    
    #split_row_values = files_row[merged_cols].split()
    split_row_values = vals.split(' - ')
    print(f"split_row_values: {split_row_values}")

    for idx, sep in enumerate(separated_subcols):
        if idx < 4 and sep_marker[sep] in split_row_values[idx]:
            new_vals[sep] = split_row_values[idx].replace(sep_marker[sep], '')
        elif idx == 4:
            if sep_marker[sep] in split_row_values[-1]:
                new_vals[sep] = split_row_values[-1].split(sep_marker[separated_subcols[idx+1]])[0].replace(sep_marker[sep], '')
            # there are cases when there is no time
            else:
                print(f"{sep_marker[sep]} not in {split_row_values[-1]}! (for idx=4) Setting None instead")
                new_vals[sep] = None
        elif idx == 5 and sep_marker[sep] in split_row_values[-1]:
            new_vals[sep] = split_row_values[-1].split(sep_marker[sep])[1].split(' ')[0]
        else:
            new_vals[sep] = split_row_values[-1].split(' ')[-1]  
    
    return new_vals

In [ ]:
#def split_values(files_row, sep_maker=sep_marker, merged_cols=merged_cols):
def split_values(vals, sep_marker=sep_marker, separated_subcols=separated_subcols, merged_cols_name=merged_cols_name):
    new_vals = {}
    
    #split_row_values = files_row[merged_cols].split()
    split_row_values = vals.split(' - ')
    print(f"split_row_values: {split_row_values}")

    for idx, sep in enumerate(separated_subcols):
        # not all rows have all values, so also iterate on them to prevent mishaps. 
        for split_idx, row_val in enumerate(split_row_values):
            if idx<4 and sep_marker[sep] in row_val:
                #if "Heure" not in split_row_values[split_idx+1]:
                #    print(f"row_val={row_val} - curr_sep {sep_marker[sep]}, next_sep {sep_marker[separated_subcols[idx+1]]} in {row_val}! (for idx={idx}) Setting {new_vals[sep]}")
                if not any(sep_marker[s] in row_val for s in separated_subcols[idx+1:]):
                    new_vals[sep] = row_val.replace(sep_marker[sep], '')
                    # as soon as a match was found, continue with next column separator
                    break
                else:
                    for s in separated_subcols:
                        if sep_marker[s] in row_val:
                            # if the next divider is found in the current row val, we also need to split before we fully separate
                            new_vals[sep] = row_val.split(sep_marker[s])[0].replace(sep_marker[sep], '')
                            print(f"row_val={row_val} - curr_sep {sep_marker[sep]}, sep_present {sep_marker[s]} in {row_val}! (for idx={idx}) Setting {new_vals[sep]}")
                            break
                    if sep in new_vals:
                        break
            # column separators 4 to 6 are all in the same split row value (the last one), skip until we reach it
            elif idx >= 3 and row_val==split_row_values[-1]:
                if idx ==4:
                    if sep_marker[sep] in row_val:
                        new_vals[sep] = row_val.split(sep_marker[separated_subcols[idx+1]])[0].replace(sep_marker[sep], '')
                    # there are cases when there is no time
                    else:
                        #print(f"{sep_marker[sep]} not in {row_val}! (for idx=4) Setting None instead")
                        new_vals[sep] = None
                elif idx == 5 and sep_marker[sep] in split_row_values[-1]:
                    new_vals[sep] = row_val.split(sep_marker[sep])[1].split(' ')[0]
                else:
                    new_vals[sep] = row_val.split(' ')[-1]  
            else:
                #skip row values until we reach it
                continue
    
    return new_vals

In [104]:
print(list(sep_marker[s] for s in separated_subcols[:-1]))

['Titre: ', 'Titre collection:', 'Date:', 'Chaine:', 'Heure:', ' comment= notice ']


In [112]:
def split_values(vals, sep_marker=sep_marker, separated_subcols=separated_subcols, merged_cols_name=merged_cols_name):
    new_vals = {}
    
    #split_row_values = files_row[merged_cols].split()
    split_row_values = vals.split(' - ')
    print(f"split_row_values: {split_row_values}")

    for idx, sep in enumerate(separated_subcols):
        # not all rows have all values, so also iterate on them to prevent mishaps. 
        for split_idx, row_val in enumerate(split_row_values):
            other_seps_in_row_val = any(sep_marker[s] in row_val for s in separated_subcols[idx+1:-1])
            if idx<5 and sep_marker[sep] in row_val:
                if not other_seps_in_row_val:
                #if "Heure" not in split_row_values[split_idx+1]:
                #    print(f"row_val={row_val} - curr_sep {sep_marker[sep]}, next_sep {sep_marker[separated_subcols[idx+1]]} in {row_val}! (for idx={idx}) Setting {new_vals[sep]}")
                    new_vals[sep] = row_val.replace(sep_marker[sep], '')
                    # as soon as a match was found, continue with next column separator
                    break
                else:
                    # find the first separator which is also in the row_val
                    for s in separated_subcols[idx+1:-1]:
                        if sep_marker[s] in row_val:
                            # if the next divider is found in the current row val, we also need to split before we fully separate
                            new_vals[sep] = row_val.split(sep_marker[s])[0].replace(sep_marker[sep], '').strip()
                            print(f"row_val={row_val} -- curr_sep {sep_marker[sep]}, found sep: {sep_marker[s]} in row_val! (for idx={idx}) Setting {new_vals[sep]}")
                            break
                    if sep in new_vals:
                        # as soon as a match was found, continue with next column separator
                        break
                    else:
                        # there are cases when a separator and its value are missing
                        print(f"other_seps_in_row_val is {other_seps_in_row_val} but no other separator was found in {row_val}! (for idx={idx})")
            # column separators 4 to 6 are all in the same split row value (the last one), skip until we reach it
            elif row_val == split_row_values[-1] and sep_marker[sep] in split_row_values[-1]:
                if idx == 5:
                    #print(row_val)
                    new_vals[sep] = row_val.split(sep_marker[sep])[1].split(' ')[0]
                elif idx == 6:
                    new_vals[sep] = row_val.split(' ')[-1]  
            else:
                #skip row values until we reach it
                continue

    for sep in separated_subcols:
        if sep not in new_vals:
            # set all missing separators to None
            print(f"{sep} was not found in {split_row_values} - set to None")
            new_vals[sep] = None
    
    return new_vals

In [73]:
split_row_values = example_val.split(' - ')
split_row_values

["Titre: L'Automobile en Europe",
 "Titre collection:C'est en France  c'est en Europe",
 'Date:06/10/1996',
 'Chaine:Radio France',
 'Heure:12:05:00 comment= notice 00172961 06/10/1996']

In [74]:
split_values(example_val)

split_row_values: ["Titre: L'Automobile en Europe", "Titre collection:C'est en France  c'est en Europe", 'Date:06/10/1996', 'Chaine:Radio France', 'Heure:12:05:00 comment= notice 00172961 06/10/1996']


{'Titre propre': "L'Automobile en Europe",
 'Titre collection': "C'est en France  c'est en Europe",
 'date de diffusion': '06/10/1996',
 'Chaine de diffusion': 'Radio France',
 'Heure de diffusion': '12:05:00',
 'Notice': '00172961',
 'Date de diffusion': '06/10/1996'}

In [75]:
split_values(example_val_2)

split_row_values: ['Titre: Inter soir 19h00 du 24 novembre 1988', 'Titre collection:Inter actualites de 19H00', 'Date:24/11/1988', 'Chaine:Radio France', 'Heure:19:00:00 comment= notice 00328809 24/11/1988']


{'Titre propre': 'Inter soir 19h00 du 24 novembre 1988',
 'Titre collection': 'Inter actualites de 19H00',
 'date de diffusion': '24/11/1988',
 'Chaine de diffusion': 'Radio France',
 'Heure de diffusion': '19:00:00',
 'Notice': '00328809',
 'Date de diffusion': '24/11/1988'}

In [80]:
def split_row_vals(val, old=False):
    
    if old:
        split_values_for_row = split_values_old(val)
    else:
        split_values_for_row = split_values(val)

    return split_values_for_row

In [81]:
split_values(example_val_2)

split_row_values: ['Titre: Inter soir 19h00 du 24 novembre 1988', 'Titre collection:Inter actualites de 19H00', 'Date:24/11/1988', 'Chaine:Radio France', 'Heure:19:00:00 comment= notice 00328809 24/11/1988']


{'Titre propre': 'Inter soir 19h00 du 24 novembre 1988',
 'Titre collection': 'Inter actualites de 19H00',
 'date de diffusion': '24/11/1988',
 'Chaine de diffusion': 'Radio France',
 'Heure de diffusion': '19:00:00',
 'Notice': '00328809',
 'Date de diffusion': '24/11/1988'}

In [ ]:
files_df_sep = files_df.copy()

# Apply and extract into specific columns
split_results = files_df_sep[merged_cols_name].apply(lambda x: split_row_vals(x))
for col_name in ['Titre propre', 'Titre collection', 'Chaine de diffusion', 'Heure de diffusion', 'Notice', 'Date de diffusion']:
    files_df_sep[col_name] = split_results.apply(lambda x: x.get(col_name))

files_df_sep

In [114]:
output_processed_df_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/INA/ListeDesFichiers_extended.csv"

files_df_sep.to_csv(output_processed_df_filepath)

### Match the notices to the files

#### First we need to process the notice DF, to group it by "show" (here = "Titre collection" it appears), while extracting min, max years and number of notices per collection.

When available, use "Date de diffusion" for the year, otherwise "Date d'enregistrement", check if NaNs remain

In [5]:
notice_df.head()

,Identifiant de la notice,Thèque,Titre collection,Titre propre,Date de diffusion,Date d'enregistrement,Heure de diffusion,Titre phonogramme,Genre,Thématique,Durée
0,PHD85000118,PH (Phono),Le journal sonore de la semaine,Joseph GOEBBELS lit un texte d'Adolf HITLER su...,15/03/1939,15/03/1939,NaN,NaN,Déclaration ;,Politique ;,00:03:46
1,PHD85000121,PH (Phono),Radio journal de France,Léon Blum fait appel aux détenteurs de capitaux,17/07/1936,17/07/1936,NaN,NaN,Journal parlé ;,NaN,00:13:21
2,PHD85000672,PH (Phono),La voix de Paris,Gaston HENRY-HAYE : les projets de la nouvelle...,NaN,20/10/1935,NaN,NaN,Interview entretien ; Journal parlé ;,Tourisme ;,00:02:27
3,PHD85000963,PH (Phono),Radio journal de France,Maurice BOURDET : lecture des informations du ...,07/01/1936,07/01/1936,NaN,NaN,Journal parlé ; Papier ;,Information ;,00:05:57
4,PHD85001640,PH (Phono),Radio journal de France,Appel à la nation de Jules ROMAINS,30/10/1938,30/10/1938,NaN,NaN,Déclaration ; Journal parlé ;,Littérature ; Politique ;,00:25:00


In [6]:
notice_df['Date de diffusion'].isnull().any(), notice_df["Date d'enregistrement"].isnull().any()

(np.True_, np.False_)

In [7]:
%%timeit
datetime.strptime(notice_df['Date de diffusion'][1], "%d/%m/%Y").year

57.4 μs ± 492 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [38]:
%%timeit
int(notice_df['Date de diffusion'][1].split('/')[-1])

34.1 μs ± 1.77 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [8]:
notice_df_cp = notice_df.copy()

notice_df_cp['year'] = notice_df_cp[['Date de diffusion', "Date d'enregistrement"]].apply(lambda x: int(x['Date de diffusion'].split('/')[-1]) if not pd.isna(x['Date de diffusion']) else int(x["Date d'enregistrement"].split('/')[-1]), axis=1)

print(notice_df_cp.isnull().any())

notice_df_cp.head()

Identifiant de la notice    False
Thèque                      False
Titre collection             True
Titre propre                False
Date de diffusion            True
Date d'enregistrement       False
Heure de diffusion           True
Titre phonogramme            True
Genre                        True
Thématique                   True
Durée                        True
year                        False
dtype: bool


,Identifiant de la notice,Thèque,Titre collection,Titre propre,Date de diffusion,Date d'enregistrement,Heure de diffusion,Titre phonogramme,Genre,Thématique,Durée,year
0,PHD85000118,PH (Phono),Le journal sonore de la semaine,Joseph GOEBBELS lit un texte d'Adolf HITLER su...,15/03/1939,15/03/1939,NaN,NaN,Déclaration ;,Politique ;,00:03:46,1939
1,PHD85000121,PH (Phono),Radio journal de France,Léon Blum fait appel aux détenteurs de capitaux,17/07/1936,17/07/1936,NaN,NaN,Journal parlé ;,NaN,00:13:21,1936
2,PHD85000672,PH (Phono),La voix de Paris,Gaston HENRY-HAYE : les projets de la nouvelle...,NaN,20/10/1935,NaN,NaN,Interview entretien ; Journal parlé ;,Tourisme ;,00:02:27,1935
3,PHD85000963,PH (Phono),Radio journal de France,Maurice BOURDET : lecture des informations du ...,07/01/1936,07/01/1936,NaN,NaN,Journal parlé ; Papier ;,Information ;,00:05:57,1936
4,PHD85001640,PH (Phono),Radio journal de France,Appel à la nation de Jules ROMAINS,30/10/1938,30/10/1938,NaN,NaN,Déclaration ; Journal parlé ;,Littérature ; Politique ;,00:25:00,1938


#### Small digression to find all the broadcast types in the collection

In [ ]:
notice_df_cp['Type emission'] = notice_df_cp["Genre"].apply(lambda x: [str(y).strip() for y in x.split(';') if str(y).strip()!=''] if pd.notna(x) else [])

possible_types = Counter([t for v in notice_df_cp['Type emission'].values for t in v ])
possible_types

Counter({'Journal parlé': 21416,
         'Magazine': 9454,
         'Reportage': 3897,
         'Causerie': 3787,
         'Interview entretien': 2550,
         'Débat': 2530,
         'Papier': 1718,
         'Déclaration': 983,
         'Interview': 252,
         'Bruitage image sonore': 185,
         'Chronique': 152,
         'Lecture': 96,
         'Interprétation': 94,
         'Récit portrait': 88,
         'Entretien': 88,
         'Elément brut': 30,
         'Retransmission': 28,
         'Documentaire': 27,
         'Réalisation dans un lieu public': 26,
         'Conférence de presse': 14,
         'Evocation scénarisée': 12,
         'Indicatif': 11,
         'Emission à base de disques': 8,
         'Témoignage': 7,
         "Document à base d'archives": 6,
         'Rétrospective': 5,
         'Jeu': 5,
         'Feuilleton': 5,
         'Brève': 3,
         'Micro trottoir': 3,
         'Message info': 1,
         'Création sonore': 1,
         'Revue de presse': 1,
  

Find the notices which don't have a collection title

In [96]:
collectionless_df = notice_df_cp[notice_df_cp["Titre collection"].isnull()]
collectionless_df

,Identifiant de la notice,Thèque,Titre collection,Titre propre,Date de diffusion,Date d'enregistrement,Heure de diffusion,Titre phonogramme,Genre,Thématique,Durée,year
227,PHD85000827,PH (Phono),NaN,Philippe HENRIOT : compte rendu de la conféren...,NaN,14/12/1942,NaN,NaN,Journal parlé ;,Politique ;,00:07:03,1942
228,PHD85005840,PH (Phono),NaN,Commandant DUVIVIER : la situation de la Radio...,NaN,09/10/1941,NaN,NaN,Journal parlé ;,Média ;,00:04:26,1941
229,PHD85005989,PH (Phono),NaN,Actualité agricole et concours de la radiodiff...,NaN,22/06/1943,NaN,NaN,Journal parlé ;,Information ;,00:04:58,1943
230,PHD85007219,PH (Phono),NaN,Déclaration de François DARLAN lue par Evrard,NaN,23/05/1941,NaN,NaN,Journal parlé ; Lecture ;,Information ;,00:03:57,1941
231,PHD85007536,PH (Phono),NaN,Joseph DARNAND : message aux SOL,NaN,02/12/1942,NaN,NaN,Journal parlé ; Papier ;,Information ;,00:03:20,1942
232,PHD85010344,PH (Phono),NaN,Visite du Maréchal PETAIN à Paris,NaN,26/04/1944,NaN,NaN,Bruitage image sonore ; Déclaration ; Journal ...,Information ; Religion ;,00:57:58,1944
233,PHD85027579,PH (Phono),NaN,Allocution de François CHASSEIGNE sur la relèv...,NaN,04/11/1942,NaN,NaN,Causerie ; Journal parlé ;,Information ;,00:03:54,1942
234,PHD85027587,PH (Phono),NaN,Allocution de Pierre CHARBIN : le ravitailleme...,NaN,29/08/1941,NaN,NaN,Déclaration ; Journal parlé ;,Information ;,00:08:38,1941
235,PHD85027588,PH (Phono),NaN,La réorganisation du service du ravitaillement,NaN,01/01/1941,NaN,NaN,Déclaration ; Journal parlé ;,Information ;,00:01:52,1941
237,PHD85027673,PH (Phono),NaN,L'organisation du ravitaillement,NaN,01/08/1943,NaN,NaN,Interview entretien ; Journal parlé ;,Information ;,00:02:06,1943


Group the notices per show and extract start, end years as well as number of notices.

But first unify the collection names which vary with time: 
- "[year]... 2000 : Comprendre aujourd'hui pour vivre demain"
- "Inter actualités de [tile]"

Then cean a bit the Genres and themes

In [85]:
notice_df_cp["Titre collection"] = notice_df_cp["Titre collection"].apply(lambda x: "75... 2000 : Comprendre aujourd'hui pour vivre demain" if "Comprendre aujourd'hui pour vivre demain" in str(x) else x)
notice_df_cp["Titre collection"] = notice_df_cp["Titre collection"].apply(lambda x: "Inter actualités" if "Inter actualités" in str(x) else x)

In [54]:
shows_df = notice_df_cp.groupby("Titre collection").agg(
    **{
    'année_début': pd.NamedAgg(column="year", aggfunc='min'),
    'année_fin': pd.NamedAgg(column="year", aggfunc='max'),
    'Identifiants des notices': pd.NamedAgg(column="Identifiant de la notice", aggfunc=list),
    'Nombre de notices': pd.NamedAgg(column="Identifiant de la notice", aggfunc='count'),
    'Genres': pd.NamedAgg(column="Genre", aggfunc=list),
    'Thématiques': pd.NamedAgg(column="Thématique", aggfunc=list)}
).reset_index()

shows_df["Genres"] = shows_df["Genres"].apply(lambda x: list(set(x)))
shows_df["Thématiques"] = shows_df["Thématiques"].apply(lambda x: list(set(x)))

shows_df

,Titre collection,année_début,année_fin,Identifiants des notices,Nombre de notices,Genres,Thématiques
0,75... 2000 : Comprendre aujourd'hui pour vivre...,1975,1976,"[PHD95073186, PHD95073187, PHD95073188, PHD950...",9,"[Débat ; , Magazine ; ]","[Economie ; Politique ; Société ; , Education ..."
1,76... 2000 : Comprendre aujourd'hui pour vivre...,1976,1977,"[PHD98000057, PHD98000058, PHD98000059, PHD980...",46,"[Débat ; Journal parlé ; , Débat ; , Débat ; M...",[Economie ; Politique ; Sciences humaines ; So...
2,77... 2000 : Comprendre aujourd'hui pour vivre...,1977,1977,"[PHD98000102, PHD98000103, PHD98000104, PHD980...",11,[Débat ; Magazine ; ],"[Sciences humaines ; , Sciences humaines ; Vie..."
3,78... 2000 : Comprendre aujourd'hui pour vivre...,1978,1978,"[PHD98000152, PHD98000153, PHD98000154, PHD980...",7,"[Débat ; , Débat ; Magazine ; ]","[Sciences humaines ; , Economie ; Société ; , ..."
4,79... 2000 : Comprendre aujourd'hui pour vivre...,1979,1979,"[PHD98000220, PHD98000221, PHD98000222, PHD980...",7,"[Débat ; , Débat ; Magazine ; ]","[Sciences humaines ; , Economie ; Littérature ..."
5,80... 2000 : Comprendre aujourd'hui pour vivre...,1980,1980,"[PHD98000246, PHD98000251, PHD98000257, PHD980...",9,"[Débat ; , Débat ; Magazine ; ]","[Sciences humaines ; , Economie ; Société ; , ..."
6,81... 2000 : Comprendre aujourd'hui pour vivre...,1981,1981,"[PHD98000295, PHD98000296, PHD98000297, PHD980...",8,[Débat ; ],"[Economie ; Société ; , Média ; Société ; Tech..."
7,82... 2000 : Comprendre aujourd'hui pour vivre...,1982,1982,"[PHD98000342, PHD98000347, PHD98000348, PHD980...",6,"[Lecture ; Magazine ; , Débat ; ]","[Economie ; Politique ; Société ; , Politique ..."
8,83... 2000 : Comprendre aujourd'hui pour vivre...,1983,1983,[PHD98000404],1,[Débat ; Magazine ; ],[Sciences humaines ; Société ; Vie professionn...
9,84... 2000 : Comprendre aujourd'hui pour vivre...,1984,1984,"[PHY16066526, PHD98000473, PHD98000474]",3,"[Débat ; , Débat ; Magazine ; ]","[Economie ; Sciences humaines ; Société ; , Ec..."


We have a total of 46 shows, which tracks with the DSA access rights

In [218]:
collection_aliases = {
    "75... 2000 : Comprendre aujourd'hui pour vivre demain": "CAPVD",
    "Aux portes du monde": "PortesMd",
    "C'est en France, c'est en Europe": "CFCE",  # (already in ALL_MEDIA)
    "C'est en France  c'est en Europe": "CFCE",  # (already in ALL_MEDIA)
    "Ce soir en France": "SoirFr",
    "Conseil de l'Europe": "ConsEuro",
    "D'hier à aujourd'hui": "HierAuj",
    "D'hier a aujourd'hui": "HierAuj",
    "Des faits et des chiffres": "FChiffres",
    "Edition spéciale": "EdSpe",
    "Edition speciale": "EdSpe",
    "Editorial de Paul Marion": "RVichy", # --> put in that collection based on the folder it is from
    "Enquêtes et commentaires": "EnquetesEC",
    "Enquetes et commentaires": "EnquetesEC",
    "Grandes enquêtes": "GrEnquetes",
    "Grandes enquetes": "GrEnquetes",
    "Horizon": "Horizon",
    "Inter actualités": "InterActu",
    "Inter actualites": "InterActu",
    "Inter soir 19h00": "InterSoir",
    "L'Europe est pour demain": "EuroDemain",
    "L'économie et les hommes": "EcoEH",
    "L'economie et les hommes": "EcoEH",
    "La légion des volontaires français contre le bolchévisme": "LVFCB", # ADDED IN MEDIA LIST
    "La legion des volontaires francais contre le bolchevisme": "LVFCB", # ADDED IN MEDIA LIST
    "La milice française vous parle": "MFVP", # ADDED IN MEDIA LIST
    "La milice francaise vous parle": "MFVP", # ADDED IN MEDIA LIST
    "La ronde des nations": "RDN",  # (already in ALL_MEDIA)
    "La science en marche": "SciMarche",
    "La tribune du progrès": "TrProgres",
    "La tribune du progres": "TrProgres",
    "La voix de Paris": "VoixParis",
    "Le journal des sports": "JSports", # ADDED IN MEDIA LIST
    "Le journal sonore de la semaine": "JSDS",
    "Le monde contemporain": "MdContemp",
    "Le monde religieux": "MdReligieux",
    "Le progrès et la vie": "ProgresVie",
    "Le progres et la vie": "ProgresVie",
    "Le téléphone sonne": "TelSonne",
    "Le telephone sonne": "TelSonne",
    "Les Français parlent aux Français": "LFPAF",
    "Les Francais parlent aux Francais": "LFPAF",
    "Les enjeux internationaux": "EnjeuxInt",
    "Les grandes avenues de la science moderne": "GASM",
    "Magazine de l'ONU": "MagONU",
    "Magazine des sciences": "MagS",
    "Mode d'emploi de l'Europe": "MEEuro",
    "Paris vous parle":  "ParisVP",
    "Perspectives françaises": "PerspFr",
    "Perspectives francaises": "PerspFr",
    "Problèmes européens": "ProbEuro",
    "Problemes europeens": "ProbEuro",
    "Problèmes internationaux": "ProbInt",
    "Problemes internationaux": "ProbInt",
    "Production et productivité françaises": "ProdProdFr",
    "Production et productivite francaises": "ProdProdFr",
    "Radio Actualités Françaises": "RActuFR",
    "Radio Actualites Francaises": "RActuFR",
    "Radio Vichy : Fonds disques Pétain": "RVichy", # ADDED IN MEDIA LIST
    "Radio Vichy : Fonds disques Petain": "RVichy", # ADDED IN MEDIA LIST
    "Radio journal": "RVichy", # --> put in that collection based on the folder it is from
    "Radio journal de France": "RJournalFr",
    "Rue des entrepreneurs": "RueDE",
    "Sciences et techniques": "SciTech",
    "Tribune de Paris : Les hommes, les événements, les idées à l'ordre du jour": "TrParis",
    "Tribune de Paris : Les hommes  les evenements  les idees a l'ordre du jour": "TrParis",
    "Université radiophonique internationale": "URI",
    "Universite radiophonique internationale": "URI",
    np.nan: "NoCollec" # documents hors collection
}

Verify that all the aliases proposed by claude are valid

In [6]:
all(v not in ALL_MEDIA or v in PARTNER_TO_MEDIA["INA"] for v in collection_aliases.values())

True

### Constitute the final Listing of collections by adding the alias column based on this mapping 

But first unify the collection names which vary with time: 
- "[year]... 2000 : Comprendre aujourd'hui pour vivre demain"
- "Inter actualités de [tile]"

Then cean a bit the Genres and themes

In [55]:
notice_df_cp["Titre collection"] = notice_df_cp["Titre collection"].apply(lambda x: "75... 2000 : Comprendre aujourd'hui pour vivre demain" if "Comprendre aujourd'hui pour vivre demain" in str(x) else x)
notice_df_cp["Titre collection"] = notice_df_cp["Titre collection"].apply(lambda x: "Inter actualités" if "Inter actualités" in str(x) else x)

In [56]:
# unify the values in genre and thématique to have a clean result
notice_df_cp['Genre'] = notice_df_cp["Genre"].apply(lambda x: [str(y).strip() for y in x.split(';') if str(y).strip()!=''] if pd.notna(x) else [])
notice_df_cp['Thématique'] = notice_df_cp["Thématique"].apply(lambda x: [str(y).strip() for y in x.split(';') if str(y).strip()!=''] if pd.notna(x) else [])

In [57]:
notice_df_cp["Genre"].values, notice_df_cp["Thématique"].values

(array([list(['Déclaration']), list(['Journal parlé']),
        list(['Interview entretien', 'Journal parlé']), ...,
        list(['Journal parlé']), list(['Journal parlé', 'Reportage']),
        list([])], shape=(39607,), dtype=object),
 array([list(['Politique']), list([]), list(['Tourisme']), ..., list([]),
        list(['Information']), list([])], shape=(39607,), dtype=object))

Group the notices per show and extract start, end years as well as number of notices.
But some notices have no collection
Since one cannot groupby "NaN" vlaues, we need to add the alias column first.

In [59]:
notice_df_cp['alias'] = notice_df_cp["Titre collection"].apply(lambda x: collection_aliases[x] if x in collection_aliases else "NO ALIAS")

In [60]:
aliases_df = notice_df_cp.groupby("alias").agg(
    **{
    'année_début': pd.NamedAgg(column="year", aggfunc='min'),
    'année_fin': pd.NamedAgg(column="year", aggfunc='max'),
    'Titre collection': pd.NamedAgg(column="Titre collection", aggfunc=list),
    'Nombre de notices': pd.NamedAgg(column="Identifiant de la notice", aggfunc='count'),
    'Genres': pd.NamedAgg(column="Genre", aggfunc='sum'),
    'Thématiques': pd.NamedAgg(column="Thématique", aggfunc='sum'),
    'Identifiants des notices': pd.NamedAgg(column="Identifiant de la notice", aggfunc=list),}
).reset_index()

aliases_df["Genres"] = aliases_df["Genres"].apply(lambda x: list(set(x)))
aliases_df["Thématiques"] = aliases_df["Thématiques"].apply(lambda x: list(set(x)))
aliases_df["Titre collection"] = aliases_df["Titre collection"].apply(lambda x: list(set(x))[0] if len(list(set(x)))==1 else list(set(x)))

aliases_df.head()

,alias,année_début,année_fin,Titre collection,Nombre de notices,Genres,Thématiques,Identifiants des notices
0,CAPVD,1975,1984,75... 2000 : Comprendre aujourd'hui pour vivre...,107,"[Débat, Lecture, Journal parlé, Magazine]","[Politique, Média, Education pédagogie, Vie pr...","[PHD95073186, PHD95073187, PHD95073188, PHD950..."
1,CFCE,1996,1996,"C'est en France, c'est en Europe",17,"[Entretien, Reportage, Magazine]","[Politique, Tourisme, Société, Religion, Infor...","[00165368, 00167288, 00169364, 00171017, 00172..."
2,ConsEuro,1949,1960,Conseil de l'Europe,17,"[Déclaration, Chronique, Reportage, Journal pa...","[Politique, Information, Environnement]","[PHD86026894, PHD86026156, PHD86046313, PHD860..."
3,EcoEH,1962,1968,L'économie et les hommes,295,"[Débat, Magazine, Causerie, Journal parlé, Rep...","[Politique, Beaux arts, Education pédagogie, V...","[PHZ08001285, PHD94013850, PHD98010917, PHD980..."
4,EdSpe,1958,1963,Edition spéciale,624,"[Bruitage image sonore, Magazine, Lecture, Réc...","[Education pédagogie, Cinéma, Tradition ethniq...","[PHD98202592, PHD98203134, PHD98203149, PHZ070..."


Display the Genres and themes to be added to the metadata

In [ ]:
for idx, row in aliases_df.iterrows():
    print(f"Alias: {row.alias} - {row['Titre collection']}:")
    long_str = row.Thématiques[0]
    for t in row.Thématiques[1:]:
        long_str += ", " + t 
    print(f"    Thématiques: {long_str}")

Save the resulting dataframe

In [61]:
aliases_df_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/INA/collection_aliases.csv"

aliases_df.to_csv(aliases_df_filepath, index=0)

# Create the Issue Index from this 

In [115]:
output_processed_df_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/INA/ListeDesFichiers_extended.csv"

files_processed_df = pd.read_csv(output_processed_df_filepath, index_col=0)

##### the list of collection names is not exactly the same as the original collections - further checks needed - in particular immitating the approach for the notices

In [30]:
files_processed_df[files_processed_df["Titre collection"] == "Radio journal"]


,Type de notice,Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Format,Répertoire,Chemin,Titre propre,Titre collection,date de diffusion,Chaine de diffusion,Heure de diffusion,Notice,Date de diffusion
9003,EMISSION RAD.,Titre: La bataille du Pacifique editorial de...,PHD86069256,KO01406BIS_01.MP3,PHD86069256_KO01406BIS_01_0_54014_EXPORT.MP3,00:00:00:00,00:05:40:14,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,La bataille du Pacifique editorial de Rene-L...,Radio journal,15/06/1942,Radio Paris comment= notice PHD86069256 15/06/...,NaN,PHD86069256,15/06/1942
9004,EMISSION RAD.,Titre: La bataille du Pacifique editorial de...,PHD86069256,KO01406_01.MP3,PHD86069256_KO01406_01_0_54008_EXPORT.MP3,00:00:00:00,00:05:40:08,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,La bataille du Pacifique editorial de Rene-L...,Radio journal,15/06/1942,Radio Paris comment= notice PHD86069256 15/06/...,NaN,PHD86069256,15/06/1942


In [25]:
notice_df[notice_df["Titre collection"] == "Radio journal"]

,Identifiant de la notice,Thèque,Titre collection,Titre propre,Date de diffusion,Date d'enregistrement,Heure de diffusion,Titre phonogramme,Genre,Thématique,Durée
240,PHD86069256,PH (Phono),Radio journal,La bataille du Pacifique ; éditorial de René-L...,15/06/1942,15/06/1942,NaN,NaN,Journal parlé ;,Information ;,00:05:40


In [34]:
files_processed_df[files_processed_df["Répertoire"]=="04- JP de Radio nationale Vichy 1942-1943"]

,Type de notice,Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Format,Répertoire,Chemin,Titre propre,Titre collection,date de diffusion,Chaine de diffusion,Heure de diffusion,Notice,Date de diffusion
8983,EMISSION RAD.,Titre: Allocution de Joseph de LA PORTE DU THE...,PH202000305,KO00249_01.MP3,PH202000305_KO00249_01_113923_191722_EXPORT.MP3,00:11:39:23,00:19:17:22,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Allocution de Joseph de LA PORTE DU THEIL,NaN,01/11/942,Radio Diffusion Nationale comment= notice PH20...,NaN,PH202000305,01/11/942
8984,EMISSION RAD.,Titre: Archives politiques 1944 : General DITT...,PH202000460,KO00476_01.MP3,PH202000460_KO00476_01_4_1421211_EXPORT.MP3,00:00:00:04,01:42:12:11,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Archives politiques 1944 : General DITTMAR chr...,NaN,01/01/943,Radio Diffusion Nationale comment= notice PH20...,NaN,PH202000460,01/01/943
8985,EMISSION RAD.,Titre: Philippe HENRIOT : compte rendu de la c...,PHD85000827,KO00112_01.MP3,PHD85000827_KO00112_01_200301_270607_EXPORT.MP3,00:20:03:01,00:27:06:07,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Philippe HENRIOT : compte rendu de la conferen...,NaN,14/12/942,Radio Diffusion Nationale comment= notice PHD8...,NaN,PHD85000827,14/12/942
8986,EMISSION RAD.,Titre: Commandant DUVIVIER : la situation de l...,PHD85005840,KO00036_01.MP3,PHD85005840_KO00036_01_345902_392504_EXPORT.MP3,00:34:59:02,00:39:25:04,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Commandant DUVIVIER : la situation de la Radio...,NaN,09/10/941,Radio Diffusion Nationale comment= notice PHD8...,NaN,PHD85005840,09/10/941
8987,EMISSION RAD.,Titre: Commandant DUVIVIER : la situation de l...,PHD85005840,KO00250_01.MP3,PHD85005840_KO00250_01_6_41211_EXPORT.MP3,00:00:00:06,00:04:12:11,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Commandant DUVIVIER : la situation de la Radio...,NaN,09/10/941,Radio Diffusion Nationale comment= notice PHD8...,NaN,PHD85005840,09/10/941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9083,EMISSION RAD.,Titre: Audience des maires d'Ardeche Drome I...,PHD95079254,83ARC08415SN0035_02.MP3,PHD95079254_83ARC08415SN0035_02_1032810_113211...,01:03:28:10,01:13:21:15,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Audience des maires d'Ardeche Drome Isere et...,Radio Vichy : Fonds disques Petain,22/04/1944,Radio Diffusion Nationale comment= notice PHD9...,NaN,PHD95079254,22/04/1944
9084,EMISSION RAD.,Titre: Audience des amicales de marins : FAMAC...,PHD95079258,83ARC08415SN0035_01.MP3,PHD95079258_83ARC08415SN0035_01___EXPORT.MP3,:::,:::,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Audience des amicales de marins : FAMAC,Radio Vichy : Fonds disques Petain,29/04/1944,Radio Diffusion Nationale comment= notice PHD9...,NaN,PHD95079258,29/04/1944
9085,EMISSION RAD.,Titre: Audience des amicales de marins : FAMAC...,PHD95079258,83ARC08415SN0035_02.MP3,PHD95079258_83ARC08415SN0035_02_1443508_151351...,01:44:35:08,01:51:35:10,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa...,Audience des amicales de marins : FAMAC,Radio Vichy : Fonds disques Petain,29/04/1944,Radio Diffusion Nationale comment= notice PHD9...,NaN,PHD95079258,29/04/1944
9086,EMISSION RAD.,Titre: Mort du General HUNTZINGER dans un acci...,PHD98200317,83ARC08415SN0026_01.MP3,PHD98200317_83ARC08415SN0026_01_0_80606_EXPORT...,00:00:00:00,00:08:06:06,FVISIO,04- JP de Radio nationale Vichy 1942-1943,iMPRESSO\Journal_parle\04-_JP_de_Radio_nationa..

#### Apply the same logic to the list of audio files to recover the actual collections

In [119]:
print(files_processed_df.isnull().any())

Type de notice                                                                                                     False
Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion    False
Identifiant notice                                                                                                 False
Fichier source                                                                                                     False
Nom du fichier                                                                                                     False
TC IN                                                                                                              False
TC OUT                                                                                                             False
Format                                                                                                             False
Répertoire                      

In [219]:
files_processed_df_cp = files_processed_df.copy()

files_processed_df_cp['year'] = files_processed_df_cp['Date de diffusion'].apply(lambda x: int(x.split('/')[-1]))
# there are some typos in the years
files_processed_df_cp['year'] = files_processed_df_cp['year'].apply(lambda x: x+1000 if x<1000 else x)

files_processed_df_cp["Titre collection"] = files_processed_df_cp["Titre collection"].apply(lambda x: "75... 2000 : Comprendre aujourd'hui pour vivre demain" if "Comprendre aujourd'hui pour vivre demain" in str(x) else x)
files_processed_df_cp["Titre collection"] = files_processed_df_cp["Titre collection"].apply(lambda x: "Inter actualités" if "Inter actualités" in str(x) else x)
files_processed_df_cp["Titre collection"] = files_processed_df_cp["Titre collection"].apply(lambda x: "Inter actualites" if "Inter actualites" in str(x) else x)

files_processed_df_cp.head()

,Type de notice,Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Format,Répertoire,Chemin,Titre propre,Titre collection,Chaine de diffusion,Heure de diffusion,Notice,Date de diffusion,year
0,EMISSION RAD.,Titre: L'Automobile en Europe - Titre collecti...,172961,96F04705SA0005_01.MP3,00172961_96F04705SA0005_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,L'Automobile en Europe,C'est en France c'est en Europe,Radio France,12:05:00,00172961,06/10/1996,1996
1,EMISSION RAD.,Titre: Le dimanche des Europeens - Titre colle...,165368,96F04705SA0001_01.MP3,00165368_96F04705SA0001_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,Le dimanche des Europeens,C'est en France c'est en Europe,Radio France,12:05:00,00165368,08/09/1996,1996
2,EMISSION RAD.,Titre: Enseignement education chomage et emp...,167288,96F04705SA0002_01.MP3,00167288_96F04705SA0002_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,Enseignement education chomage et emploi,C'est en France c'est en Europe,Radio France,12:05:00,00167288,15/09/1996,1996
3,EMISSION RAD.,Titre: Le service militaire - Titre collection...,169364,96F04705SA0003_01.MP3,00169364_96F04705SA0003_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,Le service militaire,C'est en France c'est en Europe,Radio France,12:05:00,00169364,22/09/1996,1996
4,EMISSION RAD.,Titre: L'amour en Europe - Titre collection:C'...,171017,96F04705SA0004_01.MP3,00171017_96F04705SA0004_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,L'amour en Europe,C'est en France c'est en Europe,Radio France,12:05:00,00171017,29/09/1996,1996


Sometimes the repertoire indicates a collection but the collection title is missing - re-add it.

In [ ]:
files_processed_df_cp[files_processed_df_cp["Titre collection"].isna()]

In [221]:
missing_collection_titles = {"04- JP de Radio nationale Vichy 1942-1943": "Radio Vichy : Fonds disques Petain",
                             "12- Inter actualités 1960-1969": 'Inter actualites'}

files_processed_df_cp["Titre collection"] = files_processed_df_cp[["Répertoire", "Titre collection"]].apply(lambda x: missing_collection_titles[x["Répertoire"]] if pd.isna(x["Titre collection"]) else x["Titre collection"], axis=1)

In [222]:

files_processed_df_cp['alias'] = files_processed_df_cp["Titre collection"].apply(lambda x: collection_aliases[x] if x in collection_aliases else "NO ALIAS")

In [223]:
aliases_from_files_df = files_processed_df_cp.groupby("alias").agg(
    **{
    'année_début': pd.NamedAgg(column="year", aggfunc='min'),
    'année_fin': pd.NamedAgg(column="year", aggfunc='max'),
    'Titre collection': pd.NamedAgg(column="Titre collection", aggfunc=list),
    'Nombre de notices': pd.NamedAgg(column="Notice", aggfunc='nunique'),
    #'Genres': pd.NamedAgg(column="Genre", aggfunc='sum'),
    #'Thématiques': pd.NamedAgg(column="Thématique", aggfunc='sum'),
    'Chaine de diffusion': pd.NamedAgg(column="Chaine de diffusion", aggfunc=list),
    "Répertoire": pd.NamedAgg(column="Répertoire", aggfunc=list),
    'Identifiants des notices': pd.NamedAgg(column="Notice", aggfunc=list),}
).reset_index()

#aliases_from_files_df['Chaine de diffusion'] = aliases_from_files_df['Chaine de diffusion'].apply(lambda x: list(set(x)))
aliases_df["Identifiants des notices"] = aliases_df["Identifiants des notices"].apply(lambda x: list(set(x)))
aliases_from_files_df["Chaine de diffusion"] = aliases_from_files_df["Chaine de diffusion"].apply(lambda x: list(set(x))[0] if len(list(set(x)))==1 else list(set(x)))
aliases_from_files_df['Titre collection'] = aliases_from_files_df['Titre collection'].apply(lambda x: list(set(x))[0] if len(list(set(x)))==1 else list(set(x)))
aliases_from_files_df['Répertoire'] = aliases_from_files_df['Répertoire'].apply(lambda x: list(set(x))[0] if len(list(set(x)))==1 else list(set(x)))



aliases_from_files_df

,alias,année_début,année_fin,Titre collection,Nombre de notices,Chaine de diffusion,Répertoire,Identifiants des notices
0,CAPVD,1975,1984,75... 2000 : Comprendre aujourd'hui pour vivre...,107,Radio France,75...2000 : comprendre pour vivre demain,"[PHD95073186, PHD95073187, PHD95073188, PHD950..."
1,CFCE,1996,1996,C'est en France c'est en Europe,17,Radio France,Construction européenne,"[00172961, 00165368, 00167288, 00169364, 00171..."
2,ConsEuro,1949,1960,Conseil de l'Europe,17,Radio Television Francaise,Construction européenne,"[PHD86026156, PHD86026894, PHD86037808, PHD860..."
3,EcoEH,1962,1968,L'economie et les hommes,295,"[Office Radio Television France, Radio Televis...",Les banques suisses,"[PHD90000370, PHD94008439, PHD94008440, PHD940..."
4,EdSpe,1958,1963,Edition speciale,624,Radio Television Francaise,Edition spéciale,"[PHD88011641, PHD88011647, PHD88011650, PHD940..."
5,EnjeuxInt,1984,1996,Les enjeux internationaux,1402,Radio France,Les organisations internationales,"[00745048, 00745078, 00745091, 00745151, 00745..."
6,EnquetesEC,1958,1968,Enquetes et commentaires,1679,"[Office Radio Television France, Radio Televis...",Enquêtes et commentaires,"[PHD88012247, PHD94019542, PHD94025239, PHD940..."
7,EuroDemain,1955,1955,L'Europe est pour demain,7,Radio Television Francaise,Construction européenne,"[PHD88014523, PHD88014524, PHD88014525, PHD880..."
8,FChiffres,1958,1961,Des faits et des chiffres,150,Radio Television Francaise,Les banques suisses,"[PHD98203650, PHD98203650, PHD98203650, PHD982..."
9,GASM,1969,1987,Les grandes avenues de la science moderne,733,"[Radio France, Office Radio Television France]",Energie nucléaire et armes nucléaires,"[PHD94024229, PHD94024230, PHD94025006, PHD940..."


There a very slight differences in the results, so save them also, comparisons will be made later

In [224]:
aliases_from_files_df_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/INA/collection_aliases_from_files.csv"

aliases_from_files_df.to_csv(aliases_from_files_df_filepath)